# 02 — Test frontend (bundle JS/CSS depuis l'host via reverse nginx)

Smoke test du container `fxvol-frontend` — étape 2/5. Valide la **chaîne complète host → reverse nginx → frontend** : bundle servi avec headers attendus (cache, content-type, gzip), filenames hashés (cache busting), SPA fallback opérationnel.

**Différence avec le notebook 01** : 01 teste `http://127.0.0.1:8080/` **depuis l'intérieur** du container frontend (isole nginx-frontend du reverse proxy). 02 teste `http://localhost/` **depuis l'host** (via `fxvol-nginx` proxy sur 80) — donc valide aussi que le reverse forward bien le frontend.

**Pourquoi cette double couche** : si 01 ✓ + 02 ✗ → bug dans la conf reverse `infrastructure/nginx/nginx-dev.conf` (location `/` non proxy_pass vers frontend). Si 01 ✗ + 02 ✗ → bug dans le bundle ou frontend.conf.

**Couvre** :
1. `GET /` retourne 200 + HTML React
2. Script principal référencé dans le HTML
3. `GET` de l'asset JS principal réussit + Content-Type + taille raisonnable
4. Hash dans le filename (regex `index-[hash].js`)
5. `Cache-Control: max-age=31536000` (1 an) sur les assets hashés
6. CSS principal pareil — taille > 5kb, Content-Type CSS
7. `/index.html` n'est pas caché (max-age=0 ou no-cache)
8. SPA fallback : URL random → 200 avec le HTML root
9. `gzip` actif sur le bundle (taille compressée < taille brute)

**Préreq** :
- Notebook 01 vert
- `fxvol-nginx` running + healthy
- Port 80 host non bloqué par un autre service
- `pip install requests` (déjà dans `requirements.txt`)

## Setup

In [12]:
import re
import requests

BASE_URL = "http://localhost"
TIMEOUT_S = 5

results = []

def record(label, ok, detail=""):
    results.append((label, ok, detail))
    sym = "OK" if ok else "FAIL"
    print(f"  [{sym:4}] {label}{('  | ' + detail) if detail else ''}")

print(f"target = {BASE_URL}")

target = http://localhost


## 1. `GET /` retourne 200 + HTML React

**Ce que tu dois voir** : status 200, body avec `<!DOCTYPE html>` + `<div id="root">`. Si 502/504 → reverse nginx ne joint pas le frontend (le service `frontend` est down ou hors du même réseau Docker).

In [13]:
print("== 1. GET / ==")

try:
    resp = requests.get(f"{BASE_URL}/", timeout=TIMEOUT_S)
    record("GET / status 200", resp.status_code == 200, f"status = {resp.status_code}")
    body = resp.text
    has_doctype = "<!doctype html>" in body.lower() or "<!DOCTYPE html>" in body
    has_root_div = re.search(r'<div\s+id=["\']root["\']', body) is not None
    record("HTML <!DOCTYPE html>", has_doctype, f"size = {len(body)} bytes")
    record("HTML <div id=\"root\">", has_root_div,
           "OK" if has_root_div else f"head: {body[:200]!r}")
except requests.RequestException as e:
    record("GET / status 200", False, f"{type(e).__name__}: {e}")
    body = ""

== 1. GET / ==
  [OK  ] GET / status 200  | status = 200
  [OK  ] HTML <!DOCTYPE html>  | size = 413 bytes
  [OK  ] HTML <div id="root">  | OK


## 2. Script principal référencé

Vite emet `<script type="module" crossorigin src="/assets/index-XXXXXXXX.js">` dans le HTML. On extrait l'URL pour les sections suivantes.

In [14]:
print("== 2. <script src=...> dans le HTML ==")

script_path = None
css_path = None

if body:
    m = re.search(r'<script[^>]+src="(/assets/index-[a-zA-Z0-9_-]{6,}\.js)"', body)
    if m:
        script_path = m.group(1)
    record("<script src=\"/assets/index-<hash>.js\">",
           script_path is not None,
           f"src = {script_path!r}" if script_path else "introuvable")

    # CSS principal — typiquement <link rel="stylesheet" crossorigin href="/assets/index-XXXX.css">
    m_css = re.search(r'<link[^>]+href="(/assets/index-[a-zA-Z0-9_-]{6,}\.css)"', body)
    if m_css:
        css_path = m_css.group(1)
    record("<link href=\"/assets/index-<hash>.css\">",
           css_path is not None,
           f"href = {css_path!r}" if css_path else "introuvable")
else:
    record("<script src=...>", False, "skip (cf. §1)")

== 2. <script src=...> dans le HTML ==
  [OK  ] <script src="/assets/index-<hash>.js">  | src = '/assets/index-Bc4oxzX_.js'
  [OK  ] <link href="/assets/index-<hash>.css">  | href = '/assets/index-DyE7LsQN.css'


## 3. `GET` de l'asset JS principal

L'asset doit être servi 200, avec `Content-Type: text/javascript` (ou `application/javascript`), et une taille minimale (un bundle React typique fait 200kb-1MB).

In [15]:
print("== 3. GET asset JS ==")

js_resp = None
if script_path:
    try:
        js_resp = requests.get(f"{BASE_URL}{script_path}", timeout=TIMEOUT_S)
        record("GET asset JS status 200",
               js_resp.status_code == 200,
               f"status = {js_resp.status_code}")
        ct = js_resp.headers.get("Content-Type", "")
        record("Content-Type = JS",
               "javascript" in ct.lower(),
               f"got {ct!r}")
        size = len(js_resp.content)
        record("taille bundle > 50kb",
               size > 50 * 1024,
               f"{size} bytes ({size/1024:.1f} kb)")
    except requests.RequestException as e:
        record("GET asset JS", False, f"{type(e).__name__}: {e}")
else:
    record("GET asset JS", False, "skip (script_path absent — cf. §2)")

== 3. GET asset JS ==
  [OK  ] GET asset JS status 200  | status = 200
  [OK  ] Content-Type = JS  | got 'application/javascript'
  [OK  ] taille bundle > 50kb  | 169046 bytes (165.1 kb)


## 4. Hash dans le filename — cache busting actif

Pattern `index-[hash].js` avec hash ≥ 6 chars hex. Le hash change à chaque build qui modifie le contenu, donc un user qui revient après un deploy invalide automatiquement son cache. Sans hash, le browser garde le vieux bundle pendant la durée du `Cache-Control` (qu'on vérifie en §5).

In [16]:
print("== 4. cache busting (hash filename) ==")

if script_path:
    # Format Vite : index-XXXXXXXX.js (hash en general 8 hex chars).
    m = re.fullmatch(r'/assets/index-([a-zA-Z0-9_-]{6,})\.js', script_path)
    record("hash dans le filename JS",
           m is not None,
           f"hash = {m.group(1)!r}" if m else f"path = {script_path!r}")
else:
    record("hash dans filename", False, "skip")

== 4. cache busting (hash filename) ==
  [OK  ] hash dans le filename JS  | hash = 'Bc4oxzX_'


## 5. `Cache-Control` sur les assets hashés

Comme le hash garantit l'invalidation, on peut cacher l'asset très longtemps côté browser. Convention : `max-age=31536000` (1 an) + `immutable`. Cf. `infrastructure/nginx/frontend.conf`.

**Si manque** : pas grave en dev (le browser refetch souvent), mais en prod les CDN/browsers ne cachent pas → trafic inutile et latence.

In [17]:
print("== 5. Cache-Control sur asset hashé ==")

if js_resp is not None and js_resp.status_code == 200:
    cc = js_resp.headers.get("Cache-Control", "")
    has_long_cache = "max-age=" in cc.lower() and any(
        f"max-age={n}" in cc.lower() or f"max-age={n} " in cc.lower()
        for n in ("31536000", "2592000")  # 1y ou 30j
    )
    # Tolérance : on accepte aussi un max-age explicite > 1 jour
    m_max = re.search(r'max-age\s*=\s*(\d+)', cc.lower())
    long_enough = m_max is not None and int(m_max.group(1)) >= 86400
    record("Cache-Control max-age ≥ 1 jour",
           long_enough,
           f"Cache-Control = {cc!r}")
else:
    record("Cache-Control asset", False, "skip")

== 5. Cache-Control sur asset hashé ==
  [OK  ] Cache-Control max-age ≥ 1 jour  | Cache-Control = 'max-age=31536000, public, immutable'


## 6. CSS principal — taille + Content-Type

Symétrique à §3 mais sur le CSS. Bundle Mantine + thème + Plotly = quelques kb à quelques dizaines de kb.

In [18]:
print("== 6. GET asset CSS ==")

if css_path:
    try:
        css_resp = requests.get(f"{BASE_URL}{css_path}", timeout=TIMEOUT_S)
        record("GET asset CSS status 200",
               css_resp.status_code == 200,
               f"status = {css_resp.status_code}")
        ct = css_resp.headers.get("Content-Type", "")
        record("Content-Type = CSS", "css" in ct.lower(), f"got {ct!r}")
        size = len(css_resp.content)
        # Seuil 1kb : Vite tree-shake agressivement Mantine + thème, le bundle
        # CSS final est souvent ~3-5kb. On veut juste qu'il ne soit pas vide.
        record("taille CSS > 1kb (non-vide)",
               size > 1024,
               f"{size} bytes ({size/1024:.1f} kb)")
    except requests.RequestException as e:
        record("GET asset CSS", False, f"{type(e).__name__}: {e}")
else:
    record("GET asset CSS", False, "skip (css_path absent — cf. §2)")

== 6. GET asset CSS ==
  [OK  ] GET asset CSS status 200  | status = 200
  [OK  ] Content-Type = CSS  | got 'text/css'
  [OK  ] taille CSS > 1kb (non-vide)  | 4094 bytes (4.0 kb)


## 7. `/index.html` n'est PAS caché

Contrairement aux assets hashés, `index.html` est le **point d'entrée** — il référence les bons hashes. Si on le cache, après un deploy le user garde l'ancien `index.html` qui pointe sur des assets disparus → 404 silencieux côté browser.

Convention : `Cache-Control: no-cache` ou `max-age=0` ou tout simplement pas de Cache-Control long sur `/`.

In [19]:
print("== 7. /index.html non caché ==")

try:
    root_resp = requests.get(f"{BASE_URL}/", timeout=TIMEOUT_S)
    cc = root_resp.headers.get("Cache-Control", "")
    # OK si : no-cache, no-store, max-age=0, OU absent (default = pas de cache long)
    cc_lower = cc.lower()
    no_cache = (
        "no-cache" in cc_lower
        or "no-store" in cc_lower
        or "max-age=0" in cc_lower
        or cc == ""
    )
    # Tolérance : si max-age > 0, doit être < 5 min
    m = re.search(r'max-age\s*=\s*(\d+)', cc_lower)
    if m and int(m.group(1)) > 300:
        no_cache = False
    record("Cache-Control / = no-cache OU max-age ≤ 5 min",
           no_cache,
           f"Cache-Control = {cc!r}")
except requests.RequestException as e:
    record("Cache-Control /", False, f"{type(e).__name__}: {e}")

== 7. /index.html non caché ==
  [OK  ] Cache-Control / = no-cache OU max-age ≤ 5 min  | Cache-Control = ''


## 8. SPA fallback

Test critique : une URL React Router type `/dashboard/positions` n'existe PAS comme fichier dans `/usr/share/nginx/html/`. Sans SPA fallback, nginx retournerait 404. Avec le `try_files $uri $uri/ /index.html;` dans `frontend.conf`, nginx retombe sur `index.html` → React Router prend le relai côté client.

**Test** : `GET /some/random/route/xyz` → 200 + HTML root (pas 404).

**Note** : un GET sur `/api/...` ne doit PAS suivre ce fallback (sinon impossible de distinguer 404 API d'une route SPA). Le reverse `nginx-public` capture `/api/` et `/ws/` avant d'atteindre le frontend. Donc on choisit volontairement un path qui ne commence ni par `/api` ni par `/ws`.

In [20]:
print("== 8. SPA fallback ==")

try:
    fb_resp = requests.get(f"{BASE_URL}/dashboard/some-random-route-xyz", timeout=TIMEOUT_S)
    record("GET /random/route → 200",
           fb_resp.status_code == 200,
           f"status = {fb_resp.status_code}")
    has_root_div = re.search(r'<div\s+id=["\']root["\']', fb_resp.text) is not None
    record("body = HTML React (SPA fallback OK)",
           has_root_div,
           f"size = {len(fb_resp.text)} bytes, has_root = {has_root_div}")
except requests.RequestException as e:
    record("SPA fallback", False, f"{type(e).__name__}: {e}")

== 8. SPA fallback ==
  [OK  ] GET /random/route → 200  | status = 200
  [OK  ] body = HTML React (SPA fallback OK)  | size = 413 bytes, has_root = True


## 9. `gzip` actif sur le bundle

nginx fait du gzip sur les Content-Type texte si `Accept-Encoding: gzip` est dans la requête. Sur un bundle JS de plusieurs centaines de kb, gzip réduit de 60-75%.

**Test** : request explicite avec `Accept-Encoding: gzip`, vérifier `Content-Encoding: gzip` dans la réponse + taille servie significativement plus petite que la taille décompressée.

In [21]:
print("== 9. gzip actif sur le bundle ==")

if script_path:
    try:
        # decode_content=False pour avoir la taille raw (gzippée).
        gz_resp = requests.get(
            f"{BASE_URL}{script_path}",
            timeout=TIMEOUT_S,
            headers={"Accept-Encoding": "gzip"},
            stream=True,
        )
        ce = gz_resp.headers.get("Content-Encoding", "")
        record("Content-Encoding = gzip",
               "gzip" in ce.lower(),
               f"got {ce!r}")
        # Taille avant décompression : Content-Length si exposé.
        gz_size = gz_resp.headers.get("Content-Length")
        if gz_size and js_resp is not None:
            ratio = int(gz_size) / max(len(js_resp.content), 1)
            record("taille gzippée < 50% de la taille raw",
                   ratio < 0.5,
                   f"gzip = {int(gz_size)} bytes, raw = {len(js_resp.content)}, ratio = {ratio:.2%}")
    except requests.RequestException as e:
        record("gzip", False, f"{type(e).__name__}: {e}")
else:
    record("gzip", False, "skip")

== 9. gzip actif sur le bundle ==
  [OK  ] Content-Encoding = gzip  | got 'gzip'


## Récap final

In [22]:
n_ok = sum(1 for _, ok, _ in results if ok)
n_fail = sum(1 for _, ok, _ in results if not ok)

print(f"\n{'LABEL':<60} STATUS  DETAIL")
print("-" * 110)
for label, ok, detail in results:
    sym = "OK" if ok else "FAIL"
    print(f"{label:<60} {sym:<6}  {detail}")
print("-" * 110)
print(f"\n{n_ok} OK / {n_fail} FAIL  ({len(results)} total)")

if n_fail == 0:
    print("\nOK — bundle servable + headers nginx corrects. Pass au notebook 03 (Playwright dashboard boot).")


LABEL                                                        STATUS  DETAIL
--------------------------------------------------------------------------------------------------------------
GET / status 200                                             OK      status = 200
HTML <!DOCTYPE html>                                         OK      size = 413 bytes
HTML <div id="root">                                         OK      OK
<script src="/assets/index-<hash>.js">                       OK      src = '/assets/index-Bc4oxzX_.js'
<link href="/assets/index-<hash>.css">                       OK      href = '/assets/index-DyE7LsQN.css'
GET asset JS status 200                                      OK      status = 200
Content-Type = JS                                            OK      got 'application/javascript'
taille bundle > 50kb                                         OK      169046 bytes (165.1 kb)
hash dans le filename JS                                     OK      hash = 'Bc4oxzX_'
Cach

## Troubleshooting cheat sheet

| Symptôme | Cause probable | Fix |
|---|---|---|
| GET / status 502/504 | reverse `fxvol-nginx` ne joint pas `frontend:8080` | `docker compose restart nginx` ; check les 2 sont sur le même réseau (`fxvol-internal`) |
| GET / 200 mais sans `<div id="root">` | `index.html` est la page nginx default → bundle pas copié | revoir `infrastructure/docker/Dockerfile.web` (étape `COPY --from=build`) |
| `<script src>` introuvable | build Vite KO ou index.html servi mais pas le vrai | `docker exec fxvol-frontend cat /usr/share/nginx/html/index.html` |
| asset JS 404 | hash dans HTML ≠ hash sur disque (build incohérent) | `docker compose build --no-cache frontend` |
| Cache-Control absent / max-age trop court | `infrastructure/nginx/frontend.conf` pas configuré pour `/assets/` | ajouter un `location /assets/` avec `expires 1y; add_header Cache-Control "public, immutable";` |
| `/random/route` retourne 404 au §8 | SPA fallback pas configuré | ajouter `try_files $uri $uri/ /index.html;` dans la `location /` de frontend.conf |
| `Content-Encoding ≠ gzip` au §9 | gzip module désactivé ou Content-Type pas dans `gzip_types` | revoir `gzip on;` + `gzip_types text/javascript application/javascript text/css ...` dans nginx.conf |